# CL For Audio

Example of a complete training and test loop using the **GDUMP algorithm** for environmental sound recognition (ESR) using the **"dcase_icub"** dataset containg **four classes**. This tutorial is intended for undergraduates from different fields with no or only few audio processing and/or machine learning knowledge. It is therefore recommended to go through the first notebook explaining the audio preprocessing steps. People familiar with audio processing, (especially Mel spectrum and features derived from audio signals) may skip this part.

The full description/reference of the used GDUMP algorithm with examples from computer vision can be found on the original Avalanche website.

For an overview of how different algorithms compare to the three different ESR datasets of different scale, please refer to the paper:
**Omar Eldardeer, Francesco Rea, Giulio Sandini, and Doreen Jirak: When Deep is not Enough: Towards Understanding Shallow and Continual Learning Models in Realistic Environmental Sound Classification for Robots**

https://www.worldscientific.com/doi/abs/10.1142/S0219843623500081

## 1. Install the required libraries using pip

In [ ]:
%pip install avalanche-lib
%pip install typeguard
%pip install pandas
%pip install torch==2.7.0 torchaudio==2.7.0 torchvision==0.22.0
%pip install soundfile

Note: you may need to restart the kernel to use updated packages.


## 2. Importing Libraries

In [5]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
import soundfile as sf
import torch
import torchaudio
from torch.utils.data import Dataset
import os
from pathlib import Path

## 3. Creating your tourch audio dataset 
Before setting up a concrete model, this step is crucial to get your data in. Using Pytorch convention, you first need to create a "Dataset" class that shapes your input data and the labels (cf. also class constructor).

You can also define extra functionality to shape your data, e.g., like here, use only one channel (mono) or cut/padd your audio sequences to obtain equal lengths.

Please also use the placeholders to change the paths to your own directories, i.e. the place where your data is located.

In [53]:
class m_audio_dataset(Dataset):

    def __init__(self,
                    audio_dir,
                    transformation,
                    num_samples,
                    pd_data,
                    kind='train'):
        self.audio_dir = audio_dir
        self.num_samples = num_samples
        if (transformation):
            self.transformation = transformation

        if kind=='train':
            self.items = pd_data[pd_data['train'] == True][['recordedName','label','className']]
            self.items.reset_index(inplace = True,drop = True)
        elif kind=='test':
            self.items = pd_data[pd_data['train'] == False][['recordedName','label','className']]
            self.items.reset_index(inplace = True,drop = True)

        else:
            self.items =  pd_data[['recordedName','label','className']]
            self.items.reset_index(inplace = True,drop = True)

        self.length = len(self.items)

        self.targets = torch.as_tensor( list(self.items['label']))
    def __len__(self):
        return self.length


    def __getitem__(self, index):
        filename = self.items['recordedName'][index]
        className = self.items['className'][index]
        label = self.items['label'][index]
        signal, rate = torchaudio.load(str(self.audio_dir+ className + '/' + filename))
        signal = self._to_mono_if_necessary(signal)
        signal = self._cutting_if_necessary(signal)
        signal = self._padding_if_necessary(signal)
        data_tensor = self.transformation(signal)
        return (data_tensor, int(label))


    def _to_mono_if_necessary(self, signal):
        if signal.shape[0] > 1:
            signal = torch.mean(signal, dim=0, keepdim=True)
        return signal

    def _cutting_if_necessary(self, signal):
        if signal.shape[1] > self.num_samples:
            signal = signal[:, :self.num_samples]
        return signal

    def _padding_if_necessary(self, signal):
        length_signal = signal.shape[1]
        if length_signal < self.num_samples:
            num_missing_samples = self.num_samples - length_signal
            last_dim_padding = (0, num_missing_samples)
            signal = torch.nn.functional.pad(signal, last_dim_padding)
        return signal

## 4. Standard lines to check your GPU.

In [7]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda:0


## 5. loading the dataset. 
Here, you tell how the data is actually loaded given the parameters for the length of audio sequences, overlap, and number of Mel spectrum coefficients.

In here we use as mentioned our own dataset, which you can find here -> https://github.com/omareldardear/WhenDeepIsNotEnough 

Depending on your meata-data and how it is organized you will need to change this function to suit your dataset

In [8]:
def loadDataSet(setType="train",
                ballance=False,
                FRAME_SIZE = 1024,
                HOP_LENGTH = 512,
                N_MELS = 60,
                BASE_DIR = "",
                ANNOTATIONS_FILE = "",
                AUDIO_DIR = ""):


    pd_data = pd.read_csv(BASE_DIR + ANNOTATIONS_FILE, index_col=0)
    # adding label column
    ord_enc = OrdinalEncoder()
    pd_data["label"] = ord_enc.fit_transform(pd_data[["className"]]).astype(int)

    # splittint train test
    pd_data['train'] = False
    trainPercent = 0.7
    for labelVal in pd_data.label.unique():
        trainCount = int(len(pd_data[pd_data.label == labelVal]) * trainPercent)
        pd_data.loc[pd_data[pd_data.label == labelVal].head(trainCount).index, 'train'] = True
    file = pd_data['recordedName'][0]
    className = pd_data['className'][0]
    fullName = BASE_DIR + AUDIO_DIR + className + '/' + file
    data_array, samplerate = sf.read(fullName)


    if(ballance):
        minClassSize = min([len(pd_data[pd_data['label'] == 0]), len(pd_data[pd_data['label'] == 1]),
                            len(pd_data[pd_data['label'] == 2]), len(pd_data[pd_data['label'] == 3])])
        for i in range(4):
            pd_data = pd_data.drop(pd_data[pd_data['label'] == i].index[minClassSize:])

        pd_data = pd_data.reset_index(drop=True)
        # splittint train test
        pd_data['train'] = False
        trainPercent = 0.7
        for labelVal in pd_data.label.unique():
            trainCount = int(len(pd_data[pd_data.label == labelVal]) * trainPercent)
            pd_data.loc[pd_data[pd_data.label == labelVal].head(trainCount).index, 'train'] = True


    SAMPLES_SIZE = int(5.8 * samplerate)
    SAMPLE_RATE = samplerate

    mel_spectrogram = torchaudio.transforms.MelSpectrogram(
        sample_rate=SAMPLE_RATE,
        n_fft=FRAME_SIZE,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        normalized=True
    )

    return m_audio_dataset(AUDIO_DIR,
                            mel_spectrogram,
                            SAMPLES_SIZE,
                            pd_data,
                            kind=setType) , 4

## 6. Defining the parameters 
Number of parameters for audio processing (first block) and hyperparamters for network and training. The audio parameters are explained in the corresponding accompanying notebook.

Explanations of the hyperparameters can be found in any tutorial for neural networks.

In [42]:
config={}
config['FRAME_SIZE'] = 1024
config['HOP_LENGTH'] = 512
config['N_MELS'] = 60

config['lr_base'] = 0.1
config['wght_decay'] = 1E-05
config['momentum'] = 0.9
config['memory_size'] = 2000
config['replay_buffer_size'] = 2000
config['train_batch_size'] = 8
config['eval_batch_size'] = 8

config['batch_size'] = 32
config['nb_exp'] = 4
config['epochs'] = 2
config['lr_milestones_1'] = 49
config['lr_milestones_2'] = 63
config['lr_factor'] = 5
config['seed'] = 1234
config['modelType'] = 'icarlNet'
config['model_hs'] = 1000
config['opt'] = 'ADAM'



## 7. Getting the data instance
Here we call loadDataSet Function to get train and test datasets

................. Change the BASE DIR to the right directory of the data. 
Example data can be found

In [ ]:
BASE_DIR = "../WhenDeepIsNotEnough/dcase_icub"
ANNOTATIONS_FILE = "/recordingInfo/validRecordings.csv"
AUDIO_DIR = "/recordedAudio/"

train_data, nb_exp = loadDataSet("train",
                                 False,
                                 config['FRAME_SIZE'] ,
                                 config['HOP_LENGTH'] ,
                                 config['N_MELS'],
                                 BASE_DIR,
                                 ANNOTATIONS_FILE,
                                 AUDIO_DIR)
test_data, nb_exp = loadDataSet("test",
                                False,
                                config['FRAME_SIZE'] ,
                                config['HOP_LENGTH'] ,
                                config['N_MELS'],
                                BASE_DIR,
                                ANNOTATIONS_FILE,
                                AUDIO_DIR)


## 8. Import all the necessary Avalanche libraries

In [45]:
from avalanche.benchmarks.utils import AvalancheDataset
from avalanche.training import GDumb , ICaRL
from avalanche.models import IcarlNet, make_icarl_net, initialize_icarl_net
from avalanche.models import SimpleMLP, as_multitask
from torch.optim import SGD , Adam
from avalanche.training.plugins import EvaluationPlugin
from avalanche.evaluation.metrics import (
    ExperienceAccuracy,
    StreamAccuracy,
    EpochAccuracy,
)
from avalanche.logging.interactive_logging import InteractiveLogger
from torch.optim.lr_scheduler import MultiStepLR
from avalanche.training.plugins.lr_scheduling import LRSchedulerPlugin
import time
import pandas as pd
from torchvision import transforms

## 9. Transform your data into an Avalanche dataset.

In [46]:
train_data_avalanche = AvalancheDataset(train_data)
test_data_avalanche = AvalancheDataset(test_data)

## 10. Standard Avalanch methods

From now on, you can also refer to other tutorials in avalanch But we de in the following cells 
- "Scenario" is a Avalanche convention to set up your data to use for continual learning algorithms. 
- If you need any transformation on your input. 
- Avalanche provides plugins to log and evaluate your continual learning experiences. 
- Here, you finally set up your preferered continual learning algorithm and the network (here: MLP) that goes with it. 
- After setting up the specific model, here you declare the necessart hyperparameters necessary to run the algorithm (e.g., memory buffer size) 
- Start the training and test loop and log (meta)data, respectively, print the experiment process.

The final output will present the results of the training and test in the chosen metric (here: accuracy, cf. "evaluator" cell).

In [47]:
###  The benchmark
### ### ### ### ### ### ### ### ### ### ### ### ### ###
from avalanche.benchmarks import nc_benchmark
scenario = nc_benchmark(
        train_dataset=train_data,
        test_dataset=test_data,
        n_experiences=config["nb_exp"],
        task_labels=False,
        seed=config['seed'],
        shuffle=False,
    )


In [48]:
def transformData(inp):
    return inp

In [49]:
 ### Evaluation , plugins, and loggers
### ### ### ### ### ### ### ### ### ### ### ### ###

evaluator = EvaluationPlugin(
        EpochAccuracy(),
        ExperienceAccuracy(),
        StreamAccuracy(),
        loggers=[InteractiveLogger()],
    )

In [50]:
 ### The Model
### ### ### ### ### ### ### ### ### ### ### ### ###

if(config["modelType"] == "icarlNet"):
    model = make_icarl_net(num_classes=config['nb_exp'],n=5, c=1)
    model.apply(initialize_icarl_net)
else:

    model = SimpleMLP(num_classes=config['nb_exp'],
                        input_size=data_tensor.size()[1] * data_tensor.size()[2],
                        hidden_size=config['model_hs'],
                        drop_rate=0)

if (config['opt'] == "ADAM"):
    optim = Adam(
        model.parameters(),
        lr=config['lr_base'],
        weight_decay=config['wght_decay'],
    )



else:
    optim = SGD(
        model.parameters(),
        lr=config['lr_base'],
        weight_decay=config['wght_decay'],
        momentum=config['momentum'],
    )

sched = LRSchedulerPlugin(
    MultiStepLR(optim, [config['lr_milestones_1'], config['lr_milestones_2']], gamma=1.0 / config['lr_factor'])
)


In [51]:
### the strategy
### ### ### ### ### ### ### ### ### ### ### ### ### ###

strategy =  GDumb(
        model,
        optim,
        torch.nn.CrossEntropyLoss(),
        mem_size=config["replay_buffer_size"],
        train_mb_size=config["train_batch_size"] ,
        train_epochs=config["epochs"],
        eval_mb_size=config["eval_batch_size"],
        evaluator=evaluator,
        plugins=[sched],
        device=device)


/usr/local/src/robot/EldardeerGithub/Avalanch_colab/venv/lib/python3.12/site-packages/avalanche/training/templates/base.py:468: PositionalArgumentsDeprecatedWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your are passing 3 positional arguments to the GDumb.__init__ method. Consider passing them as names arguments. The ability to pass positional arguments will be removed in the future.
  warnings.warn(error_str, category=PositionalArgumentsDeprecatedWarning)


In [ ]:
### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ###
###  Training Code
print("Starting experiment...")
evaluationResults = []
processingTime = []
startingTime = time.time()
print("startingTime : ", startingTime)
for i, exp in enumerate(scenario.train_stream):
    eval_exps = [e for e in scenario.test_stream][: i + 1]

    print("Start training on experience ", exp.current_experience)
    expTime = time.time()
    print("starting a new exp time : ", expTime)
    val_exps = [e for e in scenario.test_stream][: i + 1]
    strategy.train(exp, num_workers=4)
    print("End training on experience", exp.current_experience)

    evaluationResults.append(strategy.eval(eval_exps, num_workers=4))
    print("Computing accuracy on the test set")
    expEndTime = time.time()
    expDuration = expEndTime - expTime
    processingTime.append(expDuration)

endTime = time.time()
print("endTime : ", endTime)
print("total Processing time :", endTime - startingTime)
procTime = endTime - startingTime


num_classes = config['nb_exp']
evaluationDfColumns = ['Exp', 'testAcc', 'trainAcc']
lst = range(0, num_classes)
evaluationDfColumns = evaluationDfColumns + ["testAccExp{:d}".format(x) for x in lst]

evaluationDf = pd.DataFrame(columns=evaluationDfColumns)
trainingDf = pd.DataFrame(columns=evaluationDfColumns)

for i in range(0, num_classes):

    evaluationLine = {}
    evaluationLine = {'Exp': i,
                        'testAcc': evaluationResults[i]['Top1_Acc_Stream/eval_phase/test_stream/Task000'],
                        'trainAcc': evaluationResults[i]['Top1_Acc_Epoch/train_phase/train_stream/Task000']}

    for j in range(0, num_classes):
        if (j <= i):
            evaluationLine["testAccExp{:d}".format(j)] = evaluationResults[i][
                "Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp{:03d}".format(j)]

    new_row = pd.Series(data=evaluationLine)

    evaluationDf = evaluationDf.append(new_row, ignore_index=True)

evaluationDf.to_csv(BaseSaveDir + "Gdumb_{:d}.csv".format(runningIdx), index=False)

print("Ending the  seed ", config["seed"])
print("last set training acc:", evaluationDf['trainAcc'].iloc[-1])
print("training acc:", evaluationDf['trainAcc'].mean())
print("testing acc:", evaluationDf['testAcc'].iloc[-1])

config['trainAccMean'] = evaluationDf['trainAcc'].mean()
config['trainAccLast'] = evaluationDf['trainAcc'].iloc[-1]
config['testAcc'] = evaluationDf['testAcc'].iloc[-1]
config['procTime'] = procTime
configDF = pd.DataFrame(config, index=[0])
configDF.to_csv(BaseSaveDir + "GDUMB_Config_{:d}.csv".format(runningIdx), index=False)

print( evaluationDf['trainAcc'].mean(), evaluationDf['trainAcc'].iloc[-1], evaluationDf['testAcc'].iloc[-1], procTime)

## Thanks for following the tutorial

### Authors: Omar Eldardeer & Doreen Jirak
####  Contact Details


Omar Eldardeer
* [Twitter](https://twitter.com/omareldardear)
* [Google Scholar](https://scholar.google.com/citations?user=2xry9p8AAAAJ&hl)
* [Linkedin](https://www.linkedin.com/in/omar-eldardear/)
* [Website](https://www.iit.it/people/omar-eldardeer )


Doreen Jirak
* [Google Scholar](https://scholar.google.com/citations?user=-HgMDDYAAAAJ&hl)
* [Linkedin](https://www.linkedin.com/in/doreen-jirak-84640b246/)
* [Website](https://www.uantwerpen.be/en/staff/doreen-jirak_26710/ )

**Happy learning! 🎉**